# 4.10 实战 Kaggle 比赛：预测房价

本节把前面学习的线性模型、优化、模型选择、正则化和数据预处理组合成一个完整的表格回归流程。任务来自 Kaggle `House Prices: Advanced Regression Techniques`：根据房屋的数值与类别属性预测销售价格。

原教材：[4.10 实战 Kaggle 比赛：预测房价](https://zh.d2l.ai/chapter_multilayer-perceptrons/kaggle-house-price.html)

> 需要提前从 Kaggle 下载 `train.csv` 和 `test.csv`。Notebook 不会自动提交结果，只会生成可上传的 `submission.csv`。

## 学习目标

1. 处理混合了数值、类别和缺失值的表格数据；
2. 理解房价比赛为何使用对数均方根误差；
3. 使用 K 折交叉验证选择超参数；
4. 在全量训练集上重新训练并生成规范的 Kaggle 提交文件；
5. 识别数据泄漏、验证集过拟合和训练/测试预处理不一致等常见错误。

## 4.10.1 竞赛与评价指标

普通均方误差会让昂贵房屋的绝对误差占据主导。比赛使用预测价格与真实价格取对数后的 RMSE：

$$\operatorname{logRMSE}=\sqrt{\frac1n\sum_{i=1}^{n}(\log \hat y_i-\log y_i)^2}.$$

它更关注相对误差。例如预测 10 万而真实 12 万，与预测 100 万而真实 120 万具有近似相同的对数误差。由于价格必须为正，计算指标时要把预测裁剪到一个很小的正数以上。

训练期间直接令网络预测 `log(SalePrice)` 通常比在原始价格尺度上优化 MSE 更稳定，也与比赛指标一致。最后生成提交文件时再用指数函数还原价格。

In [ ]:
from pathlib import Path  # 统一管理本地与 Google Drive 中的数据、图片和提交文件路径

import matplotlib.pyplot as plt  # 绘制交叉验证训练曲线
import pandas as pd  # 读取 CSV 并完成表格数据预处理
import torch  # 提供张量计算、自动微分和 GPU 支持
from torch import nn  # 提供网络层、损失函数和模型容器
from torch.utils.data import DataLoader, TensorDataset  # 将张量组织成小批量迭代器

try:  # 尝试判断当前环境是否为 Google Colab
    from google.colab import drive  # 导入 Colab 的云盘挂载工具
    drive.mount('/content/drive')  # 挂载用户的“我的云端硬盘”
    project_root = Path('/content/drive/MyDrive/d2l_learning')  # 指向云盘中的项目根目录
except ImportError:  # 本地环境没有 google.colab 时进入此分支
    project_root = Path.cwd().parent if Path.cwd().name == 'Chapter_4' else Path.cwd()  # 推断本地项目根目录

data_dir = project_root / 'data' / 'house-prices-advanced-regression-techniques'  # 设置 Kaggle 房价数据目录
figure_dir = project_root / 'Chapter_4' / 'images'  # 设置第 4 章图片保存目录
output_dir = project_root / 'Chapter_4' / 'outputs'  # 设置 Kaggle 提交文件保存目录
figure_dir.mkdir(parents=True, exist_ok=True)  # 创建图片目录且允许目录已存在
output_dir.mkdir(parents=True, exist_ok=True)  # 创建输出目录且允许目录已存在
torch.manual_seed(42)  # 固定 CPU 随机种子以复现实验结果
device = torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))  # 自动选择 Colab GPU 或本地 CPU
print('计算设备:', device)  # 输出当前实际使用的计算设备

## 4.10.2 读取并检查数据

Kaggle 训练表包含特征和目标列 `SalePrice`，测试表没有目标列。`Id` 是提交时的行标识，不应直接作为普通数值特征。若数据目录不存在，先在 Kaggle 比赛页面下载压缩包并把两个 CSV 放入下面打印的路径。

In [ ]:
train_path = data_dir / 'train.csv'  # 生成 Kaggle 训练 CSV 的完整路径
test_path = data_dir / 'test.csv'  # 生成 Kaggle 测试 CSV 的完整路径
if not train_path.exists() or not test_path.exists():  # 检查训练文件和测试文件是否都已准备好
    raise FileNotFoundError(f'请将 train.csv 和 test.csv 放到：{data_dir}')  # 缺少数据时给出明确且可操作的路径提示
train_data = pd.read_csv(train_path)  # 读取包含 SalePrice 标签的训练表
test_data = pd.read_csv(test_path)  # 读取不包含 SalePrice 标签的测试表
print('训练表形状:', train_data.shape)  # 输出训练表的行数和列数
print('测试表形状:', test_data.shape)  # 输出测试表的行数和列数
print(train_data[['Id', 'SalePrice']].head())  # 查看训练样本标识与目标价格的前五行

## 4.10.3 数据预处理

为了保证训练集和测试集拥有完全一致的特征列，先去掉训练标签与双方的 `Id`，再纵向合并后统一处理：

1. 数值特征按训练部分的均值和标准差标准化，使不同量纲更容易优化；
2. 数值缺失值填 0。标准化后 0 正好代表训练均值；
3. 类别特征使用独热编码，并把缺失值视作一个显式类别；
4. 最后再按原训练行数拆回训练特征和测试特征。

严格的交叉验证应在每一折内部估计均值、标准差和类别词表，以完全避免验证折信息进入预处理。教材式实现先联合对齐列，代码更直观，但交叉验证分数可能略显乐观；正式比赛管线宜把预处理封装到每一折中。

In [ ]:
num_train = train_data.shape[0]  # 保存训练样本数以便预处理后重新切分
all_features = pd.concat((train_data.drop(columns=['Id', 'SalePrice']), test_data.drop(columns=['Id'])), axis=0)  # 合并训练与测试特征以对齐独热编码列
numeric_columns = all_features.select_dtypes(include=['number']).columns  # 找出所有数值类型特征列
train_numeric = all_features.iloc[:num_train][numeric_columns]  # 仅取训练部分估计数值特征统计量
numeric_means = train_numeric.mean()  # 计算训练部分每个数值特征的均值
numeric_stds = train_numeric.std().replace(0, 1)  # 计算标准差并把常数列标准差替换为 1
all_features[numeric_columns] = (all_features[numeric_columns] - numeric_means) / numeric_stds  # 使用训练统计量标准化全部数值特征
all_features[numeric_columns] = all_features[numeric_columns].fillna(0)  # 用标准化后的均值 0 填充数值缺失值
all_features = pd.get_dummies(all_features, dummy_na=True, dtype='float32')  # 对类别特征和缺失类别执行独热编码
all_features = all_features.astype('float32')  # 确保所有模型输入都使用 PyTorch 友好的 float32
train_features = torch.tensor(all_features.iloc[:num_train].to_numpy(), dtype=torch.float32)  # 将训练特征转换为 CPU 张量
test_features = torch.tensor(all_features.iloc[num_train:].to_numpy(), dtype=torch.float32)  # 将测试特征转换为 CPU 张量
train_log_labels = torch.log(torch.tensor(train_data['SalePrice'].to_numpy(), dtype=torch.float32)).reshape(-1, 1)  # 将房价取对数并整理成列向量
print('处理后训练特征形状:', train_features.shape)  # 输出独热编码后的训练特征形状
print('处理后测试特征形状:', test_features.shape)  # 确认训练和测试拥有相同特征数
print('是否仍有缺失值:', torch.isnan(train_features).any().item())  # 检查处理后的训练特征是否仍含 NaN

## 4.10.4 模型、损失与训练函数

教材基线使用线性模型。独热编码后参数数量已经不少，线性基线能够检验预处理和验证流程是否正确，也便于观察权重衰减。更复杂的 MLP 未必自动更好：表格数据样本有限、特征异质，模型容量必须通过交叉验证选择。

网络直接预测对数房价，训练损失为对数空间中的 MSE；其平方根就是比赛使用的 log RMSE。

In [ ]:
def make_model(num_inputs):  # 定义根据输入特征数创建房价回归模型的函数
    model = nn.Linear(num_inputs, 1).to(device)  # 创建输出单个对数房价的线性层并移动到计算设备
    nn.init.xavier_uniform_(model.weight)  # 使用 Xavier 均匀分布初始化线性层权重
    nn.init.zeros_(model.bias)  # 将线性层偏置初始化为零
    return model  # 返回初始化完成的模型

loss_fn = nn.MSELoss()  # 创建对数房价空间的均方误差损失函数

def log_rmse(model, features, log_labels):  # 定义模型在给定数据上的对数均方根误差
    model.eval()  # 将模型切换到评估模式
    with torch.no_grad():  # 关闭梯度记录以节省评估资源
        predictions = model(features.to(device))  # 将特征移动到设备并预测对数房价
        rmse = torch.sqrt(loss_fn(predictions, log_labels.to(device)))  # 对对数空间 MSE 开平方得到 log RMSE
    return rmse.item()  # 把单元素张量转换为 Python 浮点数

def train_model(model, train_X, train_y, valid_X, valid_y, num_epochs, learning_rate, weight_decay, batch_size):  # 定义单次训练与验证流程
    train_loader = DataLoader(TensorDataset(train_X, train_y), batch_size=batch_size, shuffle=True)  # 创建随机打乱的小批量训练迭代器
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)  # 创建带权重衰减的 Adam 优化器
    history = {'train': [], 'valid': []}  # 创建保存每轮训练和验证 log RMSE 的字典
    for epoch in range(num_epochs):  # 重复遍历训练集指定轮数
        model.train()  # 将模型切换到训练模式
        for X, y in train_loader:  # 逐个小批量读取 CPU 特征与标签
            X, y = X.to(device), y.to(device)  # 将当前小批量移动到 GPU 或 CPU 计算设备
            optimizer.zero_grad()  # 清除上一批次累积在参数上的梯度
            loss = loss_fn(model(X), y)  # 前向预测对数房价并计算均方误差
            loss.backward()  # 反向传播计算模型参数梯度
            optimizer.step()  # 根据 Adam 更新规则修改模型参数
        history['train'].append(log_rmse(model, train_X, train_y))  # 记录当前轮训练集 log RMSE
        if valid_X is not None and valid_y is not None:  # 判断当前训练是否提供独立验证折
            history['valid'].append(log_rmse(model, valid_X, valid_y))  # 记录当前轮验证集 log RMSE
    return history  # 返回全部训练与验证指标历史

## 4.10.5 K 折交叉验证

一次训练/验证划分可能偶然偏难或偏易。K 折交叉验证把训练数据分成 K 份，每次用一份验证、其余 K-1 份训练，最终对 K 个验证分数取平均。每个候选超参数必须在相同折上比较，不能根据 Kaggle 测试榜反复调参。

下面通过索引切片构造折。为了公平利用全部样本，先随机排列索引，再用 `torch.tensor_split` 处理不能整除的情况。

In [ ]:
def k_fold(k, features, labels, num_epochs, learning_rate, weight_decay, batch_size):  # 定义 K 折交叉验证函数
    generator = torch.Generator().manual_seed(42)  # 创建固定种子的 CPU 随机数生成器
    shuffled_indices = torch.randperm(features.shape[0], generator=generator)  # 随机打乱全部训练样本索引
    folds = torch.tensor_split(shuffled_indices, k)  # 将打乱后的索引尽量平均分成 K 份
    train_scores, valid_scores, histories = [], [], []  # 创建保存每折分数与训练历史的列表
    for fold_index in range(k):  # 依次把每一折作为验证集
        valid_indices = folds[fold_index]  # 取出当前验证折对应的样本索引
        train_indices = torch.cat([folds[index] for index in range(k) if index != fold_index])  # 合并其余折作为训练索引
        fold_train_X, fold_train_y = features[train_indices], labels[train_indices]  # 根据索引取得当前折训练数据
        fold_valid_X, fold_valid_y = features[valid_indices], labels[valid_indices]  # 根据索引取得当前折验证数据
        model = make_model(features.shape[1])  # 为当前折创建全新且独立初始化的模型
        history = train_model(model, fold_train_X, fold_train_y, fold_valid_X, fold_valid_y, num_epochs, learning_rate, weight_decay, batch_size)  # 训练当前折模型
        train_scores.append(history['train'][-1])  # 保存当前折最终训练 log RMSE
        valid_scores.append(history['valid'][-1])  # 保存当前折最终验证 log RMSE
        histories.append(history)  # 保存当前折完整指标历史以便绘图
        print(f'折 {fold_index + 1}: 训练 log RMSE={train_scores[-1]:.4f}, 验证 log RMSE={valid_scores[-1]:.4f}')  # 输出当前折结果
    return sum(train_scores) / k, sum(valid_scores) / k, histories  # 返回平均分数与各折历史

k, num_epochs, learning_rate, weight_decay, batch_size = 5, 100, 0.01, 1e-4, 64  # 设置交叉验证和训练超参数
mean_train_score, mean_valid_score, fold_histories = k_fold(k, train_features, train_log_labels, num_epochs, learning_rate, weight_decay, batch_size)  # 执行五折交叉验证
print('平均训练 log RMSE:', mean_train_score)  # 输出五折平均训练误差
print('平均验证 log RMSE:', mean_valid_score)  # 输出用于模型选择的五折平均验证误差

In [ ]:
fig, axis = plt.subplots(figsize=(7, 4))  # 创建第一折训练曲线画布
axis.semilogy(fold_histories[0]['train'], label='train log RMSE')  # 用对数纵轴绘制第一折训练指标
axis.semilogy(fold_histories[0]['valid'], label='validation log RMSE')  # 用对数纵轴绘制第一折验证指标
axis.set_xlabel('epoch')  # 设置横轴为训练轮次
axis.grid(alpha=0.3)  # 添加半透明网格辅助读数
axis.legend()  # 显示训练与验证曲线图例
plt.tight_layout()  # 自动调整画布边距
figure_path = figure_dir / '4.10_kaggle_house_prices_cv.png'  # 生成交叉验证曲线图片路径
fig.savefig(figure_path, dpi=160, bbox_inches='tight')  # 保存高清图片并裁掉多余白边
print(f'图片已保存到：{figure_path}')  # 输出图片保存位置
plt.show()  # 在 Notebook 中显示第一折训练曲线

## 4.10.6 全量训练与生成提交文件

超参数通过交叉验证确定后，使用全部有标签训练样本重新训练一个新模型。测试集只用于生成预测，不参与梯度更新。提交文件必须保留测试表原始 `Id` 顺序，并包含列名 `Id` 和 `SalePrice`。

由于模型预测的是对数房价，需要使用 `torch.exp` 还原到价格尺度。若在 Kaggle 上继续迭代，应记录每次实验配置，不要仅根据公共排行榜噪声无限调参。

In [ ]:
final_model = make_model(train_features.shape[1])  # 使用选定结构创建最终全量训练模型
final_history = train_model(final_model, train_features, train_log_labels, None, None, num_epochs, learning_rate, weight_decay, batch_size)  # 在全部训练数据上重新训练
final_model.eval()  # 将最终模型切换到确定性的评估模式
with torch.no_grad():  # 关闭梯度记录以节省测试预测资源
    test_log_predictions = final_model(test_features.to(device))  # 在计算设备上预测测试集对数房价
    test_predictions = torch.exp(test_log_predictions).cpu().reshape(-1).numpy()  # 还原原始价格并转换为 NumPy 一维数组
submission = pd.DataFrame({'Id': test_data['Id'], 'SalePrice': test_predictions})  # 按 Kaggle 要求构造两列提交表
submission_path = output_dir / 'submission.csv'  # 生成提交 CSV 的完整保存路径
submission.to_csv(submission_path, index=False)  # 保存提交文件且不写入额外行索引
print('最终训练 log RMSE:', final_history['train'][-1])  # 输出最终模型在全量训练集上的误差
print(f'提交文件已保存到：{submission_path}')  # 输出可上传到 Kaggle 的文件位置
print(submission.head())  # 查看提交文件前五行以检查列名与数值

## 4.10.7 如何继续改进

- 在每一折内部拟合预处理统计量，构建严格无泄漏的验证管线；
- 对长尾数值特征使用 `log1p`、稳健缩放或异常值处理；
- 交叉验证学习率、权重衰减、训练轮数和模型宽度，而不是依赖单次划分；
- 尝试带非线性的 MLP、模型集成，或适合表格数据的梯度提升树，并用相同折比较；
- 检查残差与特征的关系，理解模型在哪类房屋上系统性高估或低估。

复杂模型只有在稳定交叉验证中持续优于简单基线时才值得采用。Kaggle 排名是反馈信号，但不能替代可靠的实验设计。

## 4.10.8 小结与练习答案

- 表格任务的预处理、验证设计和数据泄漏控制与模型本身同样重要；
- 对数 RMSE 更接近相对误差，直接预测对数价格能让优化目标一致；
- K 折交叉验证降低单次划分偶然性，是选择超参数的核心工具；
- 确定超参数后应使用全部训练数据重训，再对测试数据生成提交。

**练习 1：为什么不能对训练集和测试集分别独热编码？** 两边出现的类别不同会造成列数或列语义不一致，模型参数无法正确对应。

**练习 2：为何训练误差不能代替交叉验证误差？** 更复杂模型往往能降低训练误差，但目标是未见样本性能；只有独立验证折才能估计泛化。

**练习 3：为什么生成提交前要全量重训？** 交叉验证阶段每个模型只看到了部分训练数据；确定超参数后，全量重训能利用所有已知标签。